# Multi RAG Evaluator

## Load Libraries

In [ ]:
import os            # Work with environment variables and file paths
import glob          # Find files using wildcard patterns
import subprocess    # Run external commands or shell processes
import re            # Regular expressions for text pattern matching
from pathlib import Path  # file system paths
import requests      # Send HTTP requests to APIs or websites
import numpy as np   # Numerical computing (arrays, math operations)
from sklearn.manifold import TSNE   # Dimensionality reduction for visualization
import plotly.graph_objects as go   # Interactive plotting
from tqdm import tqdm   # Display progress bars for loops
from pydantic import BaseModel, Field   # Define structured data models with validation
from dotenv import load_dotenv    # Load environment variables from a .env file
from openai import OpenAI        # Official OpenAI client
from litellm import completion   # Lightweight LLM API wrapper
from langchain_ollama import ChatOllama # Ollama lanchain chat
from chromadb import PersistentClient   # Persistent vector store (Chroma DB)
from IPython.display import Markdown, display  # Markdown libraries 

c:\Users\REDTECH\miniconda3\envs\llm-openvoice\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Ollma Initialize

In [2]:
subprocess.Popen("ollama serve", shell=True)
requests.get("http://localhost:11434").content

b'Ollama is running'

In [3]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME                ID              SIZE      MODIFIED    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    4 weeks ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    4 weeks ago    



## Configuration

In [4]:
MODEL = "llama3.2"
DB_NAME = "vector_db"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2"

## Represet Documents and Chunks

In [5]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [6]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

In [7]:
class Chunks(BaseModel):
    chunks: list[Chunk]

## Fetch Documents

In [8]:
def fetch_documents():    
    # Define a function to fetch documents from the knowledge base

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": folder.name, "source": file.as_posix(), "text": f.read()})
    
    print(f"Loaded {len(documents)} documents")
    return documents

In [9]:
documents = fetch_documents()

Loaded 76 documents


In [10]:
documents[0]

{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

## Chunking Prompt

In [11]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [12]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [13]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [14]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

## Litellm Process

In [15]:
def process_document(document):
    messages = make_messages(document)

    response = completion(
        model=f"ollama/{MODEL}",
        messages=messages,
        response_format=Chunks
    )
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [16]:
process_document(documents[0])

[Result(page_content='About Insurellm: Founding and Early Growth\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup. The company experienced rapid growth, expanding its product portfolio and reaching a peak of 200 employees in 2020.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products.\nIts first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Strategic Restructuring for Growth\n\nIn 2022-2023, Insurellm underwe

## Langchain Process

In [17]:
def process_document_lanchain(document):

    ChatOllamaModel = ChatOllama(model=MODEL)
    structured_output = ChatOllamaModel.with_structured_output(Chunks, method='json_schema')
    
    messages = make_messages(document)

    structured_output = structured_output.invoke(messages)
    reply = structured_output.chunks
    return [chunk.as_result(document) for chunk in reply]

In [18]:
process_document_lanchain(documents[0])

[Result(page_content="Insurellm Founding and Early Growth\n\nThis chunk describes Insurellm's founding in 2015 by Avery Lancaster as an insurance tech startup, its early growth, and the launch of its first product Markellm.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.", metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Strategic Restructuring and Remote-First Strategy\n\nThis chunk discusses the strategic restructuring in 2022-2